# `ObsFile` observation file class sandbox

Sandbox to develop and test the `ObsFile` observation file handling class for DEMONEXT

Uses YAML (YAML Ain't Markup Language), a simple, structured, ascii-based data serialization language used
across many languages and operating systems to create human-readable and editable configuration files.

We use YAML in DEMONEXT for the obsevatory control and data-taking system runtime configuration file
and for observation specification files.

## `ObsFile` class

The `ObsFile` class implements a YAML-based observation file.  This notebook is our development and
testing sandbox before we integrate the new class into the `demonext` module.

## ObsFile format

Observation files are YAML formatted, here is a maximalist example:
<pre>
#
# DEMONEXT 2025 observation file
# Generated: 2025 Jan 29, mkObs.py v0.2.3
#
project:
  PROJECT: 'OSU_Galaxies'
  PI_NAME: 'R. Pogge'
  PI_INST: 'OSU'
  PI_EMAIL: 'pogge.1@osu.edu'
  
target:
  Name: 'NGC1234/35 Group'
  RA2000: '12:13:14.567'
  Dec2000: '+22:33:44.12'
  
sequence:
  GuideMode: none
  NumObs: 3
  Repeat: 2
  obs1: ['g_SDSS',90,3]
  obs2: ['r_SDSS',60,3]
  obs3: ['i_SDSS',120,3]

constraints:
  Airmass: 1.5
  MoonPhase: 0.5
  MoonAngle: 30.0
  MaxSky: 25000.0
  StartTime: '2025-01-26T02:04:05Z'
  EndTime: '2025-01-26T05:45:00Z'
</pre>


### Dependencies

The `obsfile.py` code requires the following:
 * YAML
 * logging
 * astropy: coordinates.SkyCoord, coordinates.TETE, units, and time.Time
 * os

### Info about YAML

 * Official YAML website: https://yaml.org/
 * PyYAML, the standard python YAML loader/emitter (comes with Anaconda): http://pyyaml.org 
 * A nice YAML tutorial: https://www.cloudbees.com/blog/yaml-tutorial-everything-you-need-get-started
 * Another YAML tutorial: https://python.land/data-processing/python-yaml


In [1]:
import obsfile

### Open and dump contents

Opens the named obs file and prints the contents to the screen by hand.

In [2]:
obsFile = "obsTest.obs"

try:
    obs = obsfile.ObsFile(obsFile)
    obs.print()
except Exception as exp:
    print(f"ERROR: loadConfig() - {exp}")



Observation file obsTest.obs contents:

project:
  PROJECT: OSU_Galaxies
  PI_NAME: R. Pogge
  PI_INST: OSU
  PI_EMAIL: pogge.1@osu.edu

target:
  OBJECT: NGC1234
  OBJCTRA: 12:13:14.567
  OBJCTDEC: +22:33:44.12

sequence:
  NumObs: 3
  obs1: ['g_SDSS', 300, 3]
  obs2: ['r_SDSS', 300, 3]
  obs3: ['i_SDSS', 360, 3]

constraints:
  Airmass: 1.5
  MoonPhase: 0.3
  MoonAngle: 30.0
  MaxSky: 25000.0
  StartTime: 2025-01-29T00:12:13Z
  EndTime: 2025-01-29T04:13:14Z


## Open and parse info

Opens `myObs.obs` and shows how to get at the contents through the class properties to 
create a summary of the observation.

### Object Coordinates

`ObsFile` expects J2000 equinox coordinates and computes the apparent RA/Dec in decimal hours/degrees,
respectively at the current instant (when the class is instantiated).  This uses a minimalist
version of the `precess()` method in the `Telescope` class, but it knows to expect sexigesimal RA/Dec.
It is reasonably agnostic provided the coordinates are RA in hours and Dec in degrees.

### Observing parameters (obsPars)

The parameters to acquire 1 or more science images though a given filter of a given exposure time
is an `obsPars` list, defined in the YAML observation file like
 > `obs1: ['g_SDSS',90,3]`

in order: `filterID`, `expTime`, and `numImages`.  The code snippet below shows how to read/parse one of these.

### code snippet

Using this with the camera object might look like this in autoguiding mode (not executable in this notebook)
```python
    import demonext
    from demonext import camera, obsfile, telescope, focuser    
    
    tel = telescope.Telescope(cfgFile)
    foc = focuser.Focuser(cfgFile)
    cam = camera.Camera(cfgFile)
    
    # ... general setup stuff
    
    # Load the observation file
    
    obsFile = "myObs.obs"
    obs = obsfile.ObsFile(obsFile)
    
    # pass project info and target name to the camera object
    
    cam.projInfo = obs.projInfo
    cam.objectID(obs.name)
    
    # point the telescope to the target ...
    
    try:
        tel.slewToRADec(obs.appRA,obs.appDec)
    except Exception as exp:
        print(f"oops: {exp}")
        # abort observation
        
    # focus the telescope ...
    
    # start the autoguider ...
    
    if cam.guidingOn():
        print(f"autoguider started...")
    else:
        print(f"ERROR: autoguider start failed: {cam.msg}")
        # abort observation
        
    # execute the observing sequence
    
    for seqNum in range(obs.repeat+1): # number of repetitions, could be 0 which means 1 sequence w/o reps
        print(f"Starting observing sequence {seqNum+1} of {obs.repeat+1}:")
        for pars in obs.obsPars:
            try:
                cam.observe(pars)  # pass pars to the Camera.observe() method
            except Exception as exp:
                print(f"oops: {exp}")
                # abort observation
    
    print("Done: observing sequence completed")
```

In [3]:
# try something

obsFile = 'myObs.obs'

try:
    obs = obsfile.ObsFile(obsFile)

    print(f"Observations of {obs.name}")
    print("\nCoordinates:")
    print(f"     J2000: {obs.RA} {obs.Dec}")
    print(f"  Apparent: {obs.appRA}h {obs.appDec}d")
    
    print(f"\nImage Sequence:")
    for i, pars in enumerate(obs.obsPars):
        filt = pars[0]
        expT = pars[1]
        numExp = pars[2]
        print(f"  Obs{i+1}: {filt} filter, {numExp}x{expT:.1f} sec")
    if obs.repeat > 0:
        print(f"  Repeat sequence {obs.repeat} times (execute {obs.repeat+1} sequences)")
    else:
        print("   No repeats")
    print(f"  GuideMode: {obs.guideMode}")

    print("\nConstraints:")
    if obs.airmass:
        print(f"  Airmass < {obs.airmass:.1f}")
    if obs.moonPhase:
        print(f"  MoonPhase < {100*obs.moonPhase:.0f}% illuminated")
    if obs.moonAngle:
        print(f"  Moon Angle < {obs.moonAngle:.1f} degrees")
    if obs.maxSky:
        print(f"  Sky Counts < {obs.maxSky:.1f} adu")
    if obs.startTime:
        print(f"  Start after {obs.startTime}")
    if obs.endTime:
        print(f"  End before {obs.endTime}")

except Exception as exp:
    print(f"ERROR: {exp}")

Observations of NGC1234/35 Group

Coordinates:
     J2000: 12:13:14.567 +22:33:44.12
  Apparent: 12.242490009839443h 22.42093537238171d

Image Sequence:
  Obs1: g_SDSS filter, 3x90.0 sec
  Obs2: r_SDSS filter, 3x60.0 sec
  Obs3: i_SDSS filter, 3x120.0 sec
  Repeat sequence 2 times (execute 3 sequences)
  GuideMode: none

Constraints:
  Airmass < 1.5
  MoonPhase < 50% illuminated
  Moon Angle < 30.0 degrees
  Sky Counts < 25000.0 adu
  Start after 2025-01-26T02:04:05Z
  End before 2025-01-26T05:45:00Z
